In [14]:
import torch
logits_teacher = torch.randn(2, 5)
_, target = torch.max(logits_teacher, dim=1)
labels_onehot = torch.zeros((target.size(0), 5))
labels_onehot[torch.arange(target.size(0)), target] = 1
print(_,target,labels_onehot)

tensor([0.8772, 1.2840]) tensor([4, 3]) tensor([[0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0.]])


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
# 忽略所有警告
warnings.filterwarnings('ignore')

def soft_entropy(input, target, reduction='mean'):
    """ Cross entropy that accepts soft targets
    Args:
         pred: predictions for neural network
         targets: targets, can be soft
         size_average: if false, sum is returned instead of mean

    Examples::

        input = torch.FloatTensor([[1.1, 2.8, 1.3], [1.1, 2.1, 4.8]])
        input = torch.autograd.Variable(out, requires_grad=True)

        target = torch.FloatTensor([[0.05, 0.9, 0.05], [0.05, 0.05, 0.9]])
        target = torch.autograd.Variable(y1)
        loss = soft_entropy(input, target)
        loss.backward()
    """
    logsoftmax = nn.LogSoftmax(dim=1)
    res = -target * logsoftmax(input)
    if reduction == 'mean':
        return torch.mean(torch.sum(res, dim=1))
    elif reduction == 'sum':
        return torch.sum(torch.sum(res, dim=1))
    else:
        return
def mix_outputs(outputs, labels, balance=False, label_dis=None):
    logits_rank = outputs[0].unsqueeze(1)
    # print('###',logits_rank)
    for i in range(len(outputs) - 1):
        logits_rank = torch.cat(
            (logits_rank, outputs[i+1].unsqueeze(1)), dim=1)
    # print('$$$',logits_rank, logits_rank.shape)###形式转换为每个样本，若干个专家的logits输出
    max_tea, max_idx = torch.max(logits_rank, dim=1) #找到每个类别在三个专家中的最大概率
    # print('&&&',max_tea, max_tea.shape)
    # min_tea, min_idx = torch.min(logits_rank, dim=1)

    non_target_labels = torch.ones_like(labels) - labels
    # print('$$$$$',non_target_labels) #1减去标签
    # tensor([[1., 0., 1.],
    #     [0., 1., 1.]])
    avg_logits = torch.sum(logits_rank, dim=1) / len(outputs)###计算每个类别在三个专家输出中的均值
    # print('^^^',avg_logits, avg_logits.shape) #可以通过这个来寻找最困难负类
    # tensor([[ 0.1688,  0.5637, -0.7943],
    #     [-0.3168,  0.5248,  0.2158]]) torch.Size([2, 3])
    # print("&&&&&&&&",avg_logits * non_target_labels, (-30 * labels))
    # tensor([[ 0.1688,  0.0000, -0.7943],
    #     [-0.0000,  0.5248,  0.2158]]) tensor([[ -0., -30.,  -0.],
    #     [-30.,  -0.,  -0.]])
    non_target_logits = (-30 * labels) + avg_logits * non_target_labels
    # print('^^^****',non_target_logits, non_target_logits.shape)
    # tensor([[  0.1688, -30.0000,  -0.7943],
    #     [-30.0000,   0.5248,   0.2158]]) torch.Size([2, 3])
    _hardest_nt, hn_idx = torch.max(non_target_logits, dim=1)
    print('###hn_idx', hn_idx, hn_idx.shape) #把找到最难非负类别的索引

    hardest_idx = torch.zeros_like(labels)
    hardest_idx.scatter_(1, hn_idx.data.view(-1, 1), 1)
    print('$$$hardest_idx',hardest_idx, hardest_idx.shape) #将最难非负类索引转换为one-hot
    hardest_logit = non_target_logits * hardest_idx
    print('***hardest_logit',hardest_logit,hardest_logit.shape) #只保留最难非负类的logist,其余变为0
    ###hn_idx tensor([0, 1]) torch.Size([2])
    # $$$hardest_idx tensor([[1., 0., 0.],
    #         [0., 1., 0.]]) torch.Size([2, 3])
    # ***hardest_logit tensor([[0.4322, -0.0000, -0.0000],
    #         [-0.0000, 0.3386, 0.0000]]) torch.Size([2, 3])
    rest_nt_logits = max_tea * (1 - hardest_idx) * (1 - labels) ###得到剩余类别中每个类别的最大概率值
    # print('aaaaaa',(1 - hardest_idx),(1 - labels))
    # tensor([[0., 1., 1.],
    #     [1., 1., 0.]]) 
    # tensor([[1., 0., 1.],
    #     [0., 1., 1.]])
    print('***rest_nt_logits',rest_nt_logits,rest_nt_logits.shape)
    # rest_nt_logits tensor([[ 0.0000,  0.0000, -0.1269],
    #         [ 0.0000,  0.0000,  0.5527]]) torch.Size([2, 3])
    reformed_nt = rest_nt_logits + hardest_logit #将最困难样本的均值logit与剩余类别在所有专家中的最大logist拼接为新的NCKD
    print('***reformed_nt',reformed_nt,reformed_nt.shape)
    # tensor([[ 0.0828,  0.0000,  1.6328],
    #     [-0.0000, -0.1575,  0.3452]]) torch.Size([2, 3])
    preds = [F.softmax(logits) for logits in outputs]  #给每个专家每个样本的输出做了个softmax
    # print('***preds',preds)
    # [tensor([[0.0481, 0.0782, 0.8737],
    #     [0.0188, 0.1820, 0.7993]]), 
    # tensor([[0.2470, 0.7355, 0.0175],
    #     [0.1796, 0.3562, 0.4643]]), 
    # tensor([[0.1659, 0.7239, 0.1102],
    #     [0.0578, 0.4596, 0.4826]])]
    reformed_non_targets = []
    for i in range(len(preds)): #将原本的概率输出中的目标类别概率变为极小值，保留非目标类别的概率值
        target_preds = preds[i] * labels

        target_preds = torch.sum(target_preds, dim=-1, keepdim=True)
        target_min = -30 * labels
        target_excluded_preds = F.softmax(
            outputs[i] * (1 - labels) + target_min)
        reformed_non_targets.append(target_excluded_preds)
    print('***reformed_non_targets',reformed_non_targets)
    # [tensor([[5.2189e-02, 1.7328e-14, 9.4781e-01],
    #     [2.5216e-14, 1.8544e-01, 8.1456e-01]]), 
    # tensor([[9.3371e-01, 2.5330e-14, 6.6288e-02],
    #     [5.0983e-14, 4.3412e-01, 5.6588e-01]]), 
    # tensor([[6.0074e-01, 4.2629e-14, 3.9926e-01],
    #     [5.3430e-14, 4.8775e-01, 5.1225e-01]])]
    label_dis = torch.tensor(
        label_dis, dtype=torch.float, requires_grad=False).cuda()
    label_dis = label_dis.unsqueeze(0).expand(labels.shape[0], -1)
    loss = 0.0
    if balance == True:
        for i in range(len(outputs)):
            loss += soft_entropy(outputs[i] + label_dis.log(), labels)
    else:
        for i in range(len(outputs)):
            # base ce
            loss += soft_entropy(outputs[i], labels)
            # hardest negative suppression
            loss += 10.0 * \
                F.kl_div(
                    torch.log(reformed_non_targets[i]), F.softmax(reformed_nt))
            # mutual distillation loss
            for j in range(len(outputs)):
                if i != j:
                    loss += F.kl_div(F.log_softmax(outputs[i]),
                                     F.softmax(outputs[j]))

    avg_output = sum(outputs) / len(outputs)
    return loss, avg_output

In [16]:
import torch
import torch.nn.functional as F

# 假设我们有5个模型的输出，每个模型的输出形状为 [batch_size, num_classes]
batch_size = 2
num_classes = 3
outputs = [torch.randn(batch_size, num_classes) for _ in range(3)]

print('outputs:',outputs)
# 真实标签，形状为 [batch_size]
#labels = torch.randint(0, num_classes, (batch_size,))
labels = torch.tensor([[0,1,0],[1,0,0]]).float()
print('labels:',labels)
# 是否使用平衡权重
balance = False

# 可选的标签分布张量，形状为 [num_classes]
label_dis = torch.tensor([0.2, 0.2, 0.6])  # 假设有一个类别的概率远高于其他类别

# 测试函数
def soft_entropy(logits, labels):
    # 这是一个示例的软熵损失函数，实际应根据需要实现
    return F.cross_entropy(logits, labels)

# 调用函数
loss, avg_output = mix_outputs(outputs, labels, balance, label_dis)

# 打印结果
print("Loss:", loss)
print("Average Output:", avg_output)

outputs: [tensor([[-0.1375, -0.1221,  0.4813],
        [-0.9834, -1.8253, -1.0195]]), tensor([[-0.9425, -2.3983,  0.0065],
        [-0.6804, -1.1185,  1.5242]]), tensor([[-0.6707, -1.5085,  0.0203],
        [-0.0671,  0.8902, -0.6860]])]
labels: tensor([[0., 1., 0.],
        [1., 0., 0.]])
###hn_idx tensor([2, 2]) torch.Size([2])
$$$hardest_idx tensor([[0., 0., 1.],
        [0., 0., 1.]]) torch.Size([2, 3])
***hardest_logit tensor([[-0.0000, -0.0000,  0.1694],
        [-0.0000, -0.0000, -0.0604]]) torch.Size([2, 3])
***rest_nt_logits tensor([[-0.1375, -0.0000,  0.0000],
        [-0.0000,  0.8902,  0.0000]]) torch.Size([2, 3])
***reformed_nt tensor([[-0.1375, -0.0000,  0.1694],
        [-0.0000,  0.8902, -0.0604]]) torch.Size([2, 3])
***reformed_non_targets [tensor([[3.5006e-01, 3.7587e-14, 6.4994e-01],
        [1.7929e-13, 3.0880e-01, 6.9120e-01]]), tensor([[2.7908e-01, 6.7021e-14, 7.2092e-01],
        [1.9026e-14, 6.6434e-02, 9.3357e-01]]), tensor([[3.3380e-01, 6.1088e-14, 6.6620e-01]